# 正确ER＋V2：历史优化逐项复测（可停止／续跑）

基准登记f8aa6a9，保留真实V2、原ER与RAW逐笔；不使用无V2/FIXED5/OHLC基准。按1→6操作，无需模型在线读日志。仅ZERO／资金费0／退出滑点0.02%。2026改为Jan1连续至北京时间11月16日00:00前，覆盖11月15日全天，11月不重置。

11项顺序及精确规则见仓库EXPERIMENT_INVENTORY.md/CSV。复用12份同区间基准与53份已核实旧账户；两项已完成FAIL/VOL从正确旧结果导入，不重算2020—2025。2026新区间是必要计算例外。旧c6d23ea错误输出拒收。未导入不填0，不静默重算。

## 1. CPU环境与必要依赖（GPU不用于金融逻辑）

In [ ]:
import sys,os,subprocess,shutil,platform
assert sys.version_info >= (3,12), '请使用Python3.12或更高的Colab运行时'
subprocess.run([sys.executable,'-m','pip','-q','install','numpy==2.3.5','pandas==3.0.1','psutil==7.0.0'],check=True)
import psutil
print('Python:',sys.version.split()[0], 'CPU:',os.cpu_count(),platform.machine())
print('RAM available/total GB:',round(psutil.virtual_memory().available/2**30,1),round(psutil.virtual_memory().total/2**30,1))
print('Local disk free GB:',round(shutil.disk_usage('/content').free/2**30,1))
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout)
else: print('GPU: none. This financial replay uses CPU.')

## 2. 固定提交与私有仓库认证
在Colab Secrets保存GITHUB_TOKEN，或使用隐藏输入。凭据不保存在Notebook、输出或remote URL。私有Notebook若打不开，先授权Colab访问GitHub，或下载.ipynb上传Colab。

In [ ]:
from pathlib import Path
import tempfile,getpass,json
CODE_COMMIT = '24a0db04419a1970f0fab7193da408f6c197adf1'
REPO = Path('/content/er-v2-retests-code-'+CODE_COMMIT[:7])
if (REPO/'.git').exists():
    assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==CODE_COMMIT, '目录提交不一致；请使用新目录，不覆盖旧任务'
else:
    if REPO.exists() and any(REPO.iterdir()): raise RuntimeError('目标目录非空；换一个新目录')
    try:
        from google.colab import userdata
        token=userdata.get('GITHUB_TOKEN')
    except Exception:
        token=getpass.getpass('GitHub private repository token (hidden): ')
    if not token: raise RuntimeError('缺少私有仓库读取凭据')
    env=os.environ.copy();env['COLAB_GH_TOKEN']=token;env['GIT_TERMINAL_PROMPT']='0'
    with tempfile.TemporaryDirectory() as tmp:
        ask=Path(tmp)/'askpass.py'
        ask.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "username" in sys.argv[1].lower() else os.environ["COLAB_GH_TOKEN"])\n')
        ask.chmod(0o700);env['GIT_ASKPASS']=str(ask)
        try:
            REPO.mkdir(parents=True,exist_ok=True)
            for cmd in [['git','init','-q'],['git','remote','add','origin','https://github.com/HITGoose/btc-eth-backtest.git'],['git','-c','credential.helper=','fetch','-q','--depth=1','origin',CODE_COMMIT],['git','checkout','-q','--detach','FETCH_HEAD']]:
                subprocess.run(cmd,cwd=REPO,env=env,check=True,capture_output=True)
        finally:
            env.pop('COLAB_GH_TOKEN',None);del token
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==CODE_COMMIT
os.environ['BTC_REPO_ROOT']=str(REPO)
ENTRY=REPO/'outputs/er_v2_retests'
print('Locked implementation:',CODE_COMMIT)
print('Repository:',REPO)

## 3. Drive、已有正确结果导入与行情缓存

先挂载Drive；失败请完成授权并重试本格，不在无备份情况下启动长回测。市场缓存继续使用原btc_eth_market_cache。本格先复制到Colab本地，计算不逐笔读取Drive。

默认从Drive `btc_eth_er_v2/ef0b074` 读取先前正确的FAIL/VOL结果。若正确ZIP另存在Drive，填写IMPORT_ZIP_PATH。不要提供旧c6d23ea的错误包。导入会核对代码指纹、配置、数据SHA、区间及现金；缺失早年结果标WAITING而不重算。2026旧Jan-Oct不能冒充Nov15完整结果。

Git有精确历史源码、基准和53份复用账本，不含全部原始行情。缺失原生／逐笔从同市场官方获取并核CHECKSUM。2026只扩展缺失日期，不用OHLC补缺口。

In [ ]:
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    try: drive.mount('/content/drive')
    except ValueError: raise RuntimeError('Drive挂载失败：请先完成账号授权、刷新并重试本格；不要跳过备份。') from None
DRIVE=Path('/content/drive/MyDrive')
DRIVE_DATA=DRIVE/'btc_eth_market_cache'
DRIVE_RESULTS=DRIVE/'btc_eth_er_v2_retests'/CODE_COMMIT[:7]
DATA=Path('/content/btc_eth_market_cache')
RESULTS=Path('/content/er_v2_retests_results_'+CODE_COMMIT[:7])
PRIOR=Path('/content/er_v2_prior_import')
IMPORT_ZIP_PATH='' # 可选：例如 /content/drive/MyDrive/results_for_review.zip

def restore(source,target,prior_only=False):
    if not source.exists(): return
    for p in source.rglob('*'):
        if not p.is_file() or p.name.endswith(('.copying','.tmp','.part','.restoring')): continue
        if prior_only and (p.suffix not in ['.json','.npz'] or 'smoke' in p.parts): continue
        dest=target/p.relative_to(source)
        if dest.exists() and dest.stat().st_mtime>=p.stat().st_mtime and dest.stat().st_size==p.stat().st_size: continue
        dest.parent.mkdir(parents=True,exist_ok=True)
        temp=dest.with_suffix(dest.suffix+'.restoring');shutil.copy2(p,temp);temp.replace(dest)
DATA.mkdir(parents=True,exist_ok=True);RESULTS.mkdir(parents=True,exist_ok=True);PRIOR.mkdir(parents=True,exist_ok=True)
restore(DRIVE_DATA,DATA);restore(DRIVE_RESULTS,RESULTS)
restore(DRIVE/'btc_eth_er_v2/ef0b074',PRIOR/'ef0b074',prior_only=True)
os.environ.update(BTC_DATA_ROOT=str(DATA),BTC_OUTPUT_ROOT=str(RESULTS),BTC_DRIVE_DATA=str(DRIVE_DATA),BTC_DRIVE_RESULTS=str(DRIVE_RESULTS),BTC_IMPORT_ROOT=str(PRIOR),PYTHONUNBUFFERED='1')
if IMPORT_ZIP_PATH:
    src=Path(IMPORT_ZIP_PATH);assert src.is_file(), 'Import ZIP path does not exist'
    local=Path('/content/corrected_prior_results.zip');shutil.copy2(src,local);os.environ['BTC_IMPORT_ZIP']=str(local)
else: os.environ.pop('BTC_IMPORT_ZIP',None)
subprocess.run([sys.executable,'-u',str(ENTRY/'import_completed.py')],cwd=REPO,check=True)
print('Local compute/cache:',DATA)
print('Drive checkpoints/results:',DRIVE_RESULTS)
print('No annual replay has started. Import details:',RESULTS/'import_status.json')
print('Recognized imported account folders:',len(list((RESULTS/'imported').glob('*/*/*/status.json'))),'; Imported count above is newly added this time.')


## 4. 最小短检查（不是年度结果）

只跑8小时、共享逐笔的独立小账本，并要求REF分钟权益与原基准对应片段完全一致。检查ATR冻结风险／单调stop、真实V2、撤单时序、阶梯语义及双币补充轻仓。输出在独立short_validation目录，不会当作年度完成。首次下载可能慢；成功后才运行5。

In [ ]:
env=os.environ.copy()
env['BTC_OUTPUT_ROOT']=str(RESULTS/'short_validation')
env['BTC_DRIVE_RESULTS']=str(DRIVE_RESULTS/'short_validation')
subprocess.run([sys.executable,'-u',str(ENTRY/'smoke.py')],cwd=REPO,env=env,check=True)
print('Only short checks complete. Full pipeline has NOT started.')

## 5. 按11项顺序正式计算

本格才启动完整计算：先补2026新范围基准，再逐项运行。每项结束保存并显示年度表，然后继续。原3参数训练网格只用2020—2022，冻结后评估；不会照搬旧无V2赢家。已有相同策略只算一次，同任务共享行情、状态隔离。

每30秒显示简短进度；缺数据或缺导入明确报告，独立项继续。停止本格会终止本次子进程组，保留最近已保存日断点。断线重新运行1→3，然后5，同提交匹配任务跳过或恢复。不改提交、不复制旧不兼容断点。费用／资金费都是0，无额外矩阵。

In [ ]:
import signal
# 可选仅降低并行数：os.environ['BTC_WORKERS']='1'
process=subprocess.Popen([sys.executable,'-u',str(ENTRY/'pipeline.py')],cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
try:
    for line in process.stdout: print(line,end='',flush=True)
    if process.wait()!=0: raise RuntimeError('Pipeline interrupted/blocked; see logs. Saved checkpoints retained.')
except KeyboardInterrupt:
    try: os.killpg(process.pid,signal.SIGTERM)
    except ProcessLookupError: pass
    process.wait()
    print('Stopped this process group. Resume same code and output root.',flush=True)
    raise
print('See incomplete.csv / import_status.json / stage_logs for any unresolved item.')

## 6. 统一汇总和下载（可导出已完成部分）

生成逐年BTC/ETH/组合结果、阶段合计、逐笔匹配、现金对账、训练分数、原因和集中度。未完成不填零，不按七期总额单独选赢家。2026无完整逐笔时仍标未完成。报告标签为描述性候选，不自动部署或组合。

下载retests_results_for_review.zip交回；若上传ZIP连接失效，至少提供REPORT.md、annual.csv、period_summary.csv、incomplete.csv、import_status.json及stage_logs。Drive也保留逐项结果和日断点。

In [ ]:
# Only read saved results; this command does NOT run pipeline.main or accounts.
script="import sys;sys.path.insert(0,"+repr(str(ENTRY))+");import common as c;r=c.module('export_retest_report',c.HERE/'report.py');r.refresh();p=c.module('export_retest_pipeline',c.HERE/'pipeline.py');p.export()"
subprocess.run([sys.executable,'-u','-c',script],cwd=REPO,check=True)
print((RESULTS/'REPORT.md').read_text())
from google.colab import files
files.download(str(RESULTS/'retests_results_for_review.zip'))